# Populate World-Camera Calibration Data

This notebook is the reproducible entry point for creating the calibration assets under `data/` from their raw recordings or primary measurement sources. It reconstructs temporary videos when video-time selections are authoritative and reads raw chunk files directly when an exact global raw-frame index is available. It does not copy previously generated calibration outputs.

Every write operation is disabled by default. Set the relevant `RUN_*` flag to `True` only after reviewing its raw source path. Existing output is preserved unless that section's `OVERWRITE_*` flag is also set to `True`. Manually chosen frames use a one-time discovery step: `chunk_io.find_frame_index` matches the current local reference TIFF to the raw chunks, records the stable global index in the measurement README, and the extraction step subsequently recreates the TIFF from that index.

Deprecated calibration artifacts, hidden operating-system files, and generated implementation caches are intentionally outside the scope of this notebook.

## Contents and provenance

| Destination | Source | Operation |
| --- | --- | --- |
| `data/exampleWorldCameraImages/` | FLIC_2001 indoor/outdoor and planetarium raw chunks | Discover the manually selected global indices, then extract the five raw frames directly from chunks |
| `data/fisheyeLensCalibration/` | Checkerboard-calibration raw chunks | Discover and extract the 47 accepted frames, then run MATLAB Camera Calibrator interactively |
| `data/flatFieldingFunction/rawFrames/` | Fels Planetarium world-camera chunks | Build a temporary video and extract the documented time samples |
| `data/radiometricCorrectionRGB/rawFrames/` | Simultaneous cloudy-sky camera recording | Extract ten raw cloudy-sky frames used with the PR670 SPD |
| `data/darkSignal/` | Covered-camera recordings at fixed AGC states | Extract ten evenly spaced dark frames per state |
| `data/empircalAGCAndIlluminance.mat` | Empirical indoor/outdoor recordings | Fit the AGC temporal model and export the selected calibration points |

The remaining root-level MAT files are documented in the final section because they originate from primary reference tables or MATLAB calibration analyses rather than world-camera chunk extraction. External measurements such as the PR670 cloudy-sky SPD cannot be recreated from camera chunks and remain explicit primary inputs.

## Shared setup and write-safety helpers

The notebook may be launched from the repository root or from this notebook's directory. The setup cell locates the repository dynamically, adds the existing Python libraries to `sys.path`, and defines narrowly scoped helpers for safe output preparation, raw-chunk index discovery, direct raw-frame extraction, and generated README provenance sections.

In [ ]:
from __future__ import annotations

import importlib
import re
import shutil
import sys
import tempfile
from pathlib import Path

import cv2
import numpy as np


def locate_project_root(start: Path | None = None) -> Path:
    """Find the repository using directories that are stable across notebook launch locations."""
    # Begin at the requested location (or the notebook's working directory) and
    # walk upward so this works whether Jupyter starts here or at the repo root.
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        # Requiring both stable top-level folders avoids accepting an unrelated
        # parent directory that happens to contain only one matching name.
        if (candidate / "data").is_dir() and (candidate / "code").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the lightLoggerAnalysis repository root.")


PROJECT_ROOT = locate_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
MATLAB_IO_PYTHON = PROJECT_ROOT / "code" / "library" / "matlabIO" / "python_libraries"
SENSOR_UTILITY_PYTHON = PROJECT_ROOT / "code" / "library" / "sensor_utility"

for library_path in (MATLAB_IO_PYTHON, SENSOR_UTILITY_PYTHON):
    if str(library_path) not in sys.path:
        sys.path.insert(0, str(library_path))

def load_video_io():
    """Import the repository video utility only when an extraction section is run.

    video_io imports MATLAB's Python engine, so deferring this import lets the
    raw-chunk and validation sections run in a plain Python kernel.
    """
    # Import lazily because only the raw-frame extraction sections need the
    # heavier video/MATLAB dependencies.
    import video_io
    return video_io


def load_chunk_io():
    """Import chunk_io only when direct raw-chunk access is required."""
    # Keeping this lazy lets the notebook documentation and validation cells run
    # even when optional chunk-processing dependencies are unavailable.
    import chunk_io
    return chunk_io


print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")

In [ ]:
def assert_safe_data_target(target: Path) -> Path:
    """Resolve a target and ensure it is below data/, never data/ itself."""
    # Resolving first prevents relative components or symlinks from bypassing
    # the containment check used by every writing helper below.
    resolved_target = target.resolve()
    resolved_data_root = DATA_ROOT.resolve()
    # A valid target must be a child of data/, not data/ itself or any path outside it.
    if resolved_target == resolved_data_root or resolved_data_root not in resolved_target.parents:
        raise ValueError(f"Refusing an unsafe output target: {resolved_target}")
    return resolved_target


def prepare_output_directory(target: Path, *, overwrite: bool) -> Path:
    """Create an empty data subdirectory, preserving existing contents unless explicitly allowed."""
    # Route all directory preparation through the common containment guard.
    target = assert_safe_data_target(target)
    target.mkdir(parents=True, exist_ok=True)
    existing_items = list(target.iterdir())
    # Refuse to mix a new extraction with old files unless replacement was requested.
    if existing_items and not overwrite:
        raise FileExistsError(
            f"{target} is not empty. Review it and set this section's OVERWRITE flag to True."
        )
    if overwrite:
        # Clear only this already-validated data subdirectory before regeneration.
        for item in existing_items:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
    return target


def write_tiff_stack(frames: np.ndarray | list[np.ndarray], output_dir: Path, *, overwrite: bool) -> None:
    """Write a frame stack as sequential, zero-based, uncompressed TIFF files."""
    # Start from an empty destination so obsolete frame numbers cannot survive.
    output_dir = prepare_output_directory(output_dir, overwrite=overwrite)
    for frame_index, frame in enumerate(frames):
        # The zero-based filename records selection order, not the source-video index.
        output_path = output_dir / f"{frame_index}.tiff"
        written = cv2.imwrite(str(output_path), frame, [cv2.IMWRITE_TIFF_COMPRESSION, 1])
        if not written:
            raise IOError(f"OpenCV could not write {output_path}")
    print(f"Wrote {len(frames)} frames to {output_dir}")


def raw_world_frame_count(raw_chunks: Path) -> int:
    """Count frames physically present in naturally ordered world chunks."""
    # chunk_io owns the filename grouping and natural ordering used throughout the repo.
    chunk_io = load_chunk_io()
    chunk_pairs = chunk_io.group_sensors_files(str(raw_chunks.resolve()))["W"]
    if not chunk_pairs:
        raise FileNotFoundError(f"No world frame chunks found in {raw_chunks}")
    # Memory mapping reads only array headers here, so counting does not load recordings.
    return sum(np.load(frame_path, mmap_mode="r").shape[0] for _, frame_path in chunk_pairs)


def extract_raw_world_frames(raw_chunks: Path, frame_indices: list[int]) -> np.ndarray:
    """Extract raw frames by zero-based global index without creating a video."""
    if not frame_indices or any(index < 0 for index in frame_indices):
        raise ValueError("frame_indices must contain nonnegative global raw-frame indices.")

    chunk_io = load_chunk_io()
    chunk_pairs = chunk_io.group_sensors_files(str(raw_chunks.resolve()))["W"]
    if not chunk_pairs:
        raise FileNotFoundError(f"No world frame chunks found in {raw_chunks}")

    # Map each requested global index to every output position that requests it.
    requested_positions: dict[int, list[int]] = {}
    for output_position, frame_index in enumerate(frame_indices):
        requested_positions.setdefault(int(frame_index), []).append(output_position)
    extracted: list[np.ndarray | None] = [None] * len(frame_indices)

    # Walk each chunk once and translate matching global indices to local offsets.
    global_offset = 0
    for _, frame_path in chunk_pairs:
        chunk_frames = np.load(frame_path, mmap_mode="r")
        chunk_stop = global_offset + len(chunk_frames)
        for frame_index, output_positions in requested_positions.items():
            if global_offset <= frame_index < chunk_stop:
                frame = np.asarray(chunk_frames[frame_index - global_offset]).copy()
                for output_position in output_positions:
                    extracted[output_position] = frame
        global_offset = chunk_stop
        if all(frame is not None for frame in extracted):
            break

    missing = [frame_indices[i] for i, frame in enumerate(extracted) if frame is None]
    if missing:
        raise IndexError(f"Raw frame indices exceed the available chunks: {missing}")
    return np.stack(extracted)


def discover_reference_frame_indices(
    reference_paths: dict[str, Path], raw_chunks_by_name: dict[str, Path]
) -> dict[str, int]:
    """Match local reference TIFFs to raw chunks and return stable indices."""
    chunk_io = load_chunk_io()
    discovered: dict[str, int] = {}
    for name, reference_path in reference_paths.items():
        # IMREAD_UNCHANGED preserves the raw two-dimensional Bayer values exactly.
        target = cv2.imread(str(reference_path), cv2.IMREAD_UNCHANGED)
        if target is None:
            raise FileNotFoundError(reference_path)
        frame_index = chunk_io.find_frame_index(
            str(raw_chunks_by_name[name].resolve()), target, verbose=True
        )
        if frame_index is None:
            raise ValueError(f"Could not find {name} in {raw_chunks_by_name[name]}")
        discovered[name] = int(frame_index)
    return discovered


def update_generated_readme_section(readme_path: Path, section_name: str, markdown: str) -> None:
    """Replace one marker-delimited provenance block in a data README."""
    # Marker replacement keeps hand-written measurement notes outside the block intact.
    readme_path = assert_safe_data_target(readme_path)
    start_marker = f"<!-- populateData:{section_name}:start -->"
    end_marker = f"<!-- populateData:{section_name}:end -->"
    text = readme_path.read_text(encoding="utf-8")
    if text.count(start_marker) != 1 or text.count(end_marker) != 1:
        raise ValueError(f"README markers for {section_name!r} are missing or duplicated.")
    before, remainder = text.split(start_marker, 1)
    _, after = remainder.split(end_marker, 1)
    replacement = f"{start_marker}\n{markdown.rstrip()}\n{end_marker}"
    readme_path.write_text(before + replacement + after, encoding="utf-8")

# `data/exampleWorldCameraImages/`

**What this is:** A small curated gallery used to inspect representative world-camera images from indoor, outdoor, and planetarium conditions. The current archive contains two indoor images, two outdoor images, and one planetarium image. All five images are completely raw, with no processing applied.

**Where it comes from:** Frames were manually selected from the FLIC_2001 `walkIndoor` and `walkOutdoor` recordings and from the planetarium recording. The visual choice is not algorithmic, so the current TIFFs serve once as reference targets for `chunk_io.find_frame_index`. The discovered zero-based raw indices are written into the directory README and become the reproducible selection.

**What it does:** The discovery flag matches each reference TIFF against its source raw chunks. The extraction flag then reads those five indexed frames directly from the chunk arrays and rewrites the TIFFs from raw data; it never copies an existing TIFF or passes the frames through a video codec.

**Output:** `data/exampleWorldCameraImages/*.tiff`.

In [ ]:
RUN_EXAMPLE_INDEX_DISCOVERY = False
RUN_EXAMPLE_FRAME_EXTRACTION = False
OVERWRITE_EXAMPLE_IMAGES = False
EXAMPLE_IMAGE_OUTPUT = DATA_ROOT / "exampleWorldCameraImages"
EXPECTED_EXAMPLE_NAMES = {
    "indoor_1.tiff",
    "indoor_2.tiff",
    "outdoor_1.tiff",
    "outdoor_2.tiff",
    "planetarium_1.tiff",
}
EXAMPLE_REFERENCE_PATHS = {name: EXAMPLE_IMAGE_OUTPUT / name for name in EXPECTED_EXAMPLE_NAMES}
EXAMPLE_RAW_CHUNKS_BY_NAME = {
    "indoor_1.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA"),
    "indoor_2.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA"),
    "outdoor_1.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA"),
    "outdoor_2.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA"),
    "planetarium_1.tiff": Path(
        "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
        "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
        "fielding_function/planetarium_fielding_function_raw"
    ),
}
EXAMPLE_SOURCE_LABELS = {
    "indoor_1.tiff": "FLIC_2001 walkIndoor",
    "indoor_2.tiff": "FLIC_2001 walkIndoor",
    "outdoor_1.tiff": "FLIC_2001 walkOutdoor",
    "outdoor_2.tiff": "FLIC_2001 walkOutdoor",
    "planetarium_1.tiff": "Fels Planetarium",
}
# Fill these values from RUN_EXAMPLE_INDEX_DISCOVERY; once recorded, they are
# sufficient to recreate the TIFFs without the reference images.
EXAMPLE_FRAME_INDICES: dict[str, int | None] = {name: None for name in EXPECTED_EXAMPLE_NAMES}


def update_example_frame_index_readme(frame_indices: dict[str, int]) -> None:
    """Record discovered example-frame indices in the measurement README."""
    rows = [
        "| Frame | Source recording | Zero-based global raw-frame index |",
        "| --- | --- | ---: |",
    ]
    for name in sorted(frame_indices):
        rows.append(f"| `{name}` | {EXAMPLE_SOURCE_LABELS[name]} | {frame_indices[name]} |")
    update_generated_readme_section(
        EXAMPLE_IMAGE_OUTPUT / "README.md", "example-frame-indices", "\n".join(rows)
    )


def populate_example_world_camera_images(
    frame_indices: dict[str, int], *, overwrite: bool = False
) -> None:
    """Recreate the five selected examples directly from their raw chunks."""
    if set(frame_indices) != EXPECTED_EXAMPLE_NAMES:
        raise ValueError("Example frame-index names do not match the curated five-frame set.")

    # Group selections by recording so each chunk sequence is traversed only once.
    frames_by_name: dict[str, np.ndarray] = {}
    for raw_chunks in sorted(set(EXAMPLE_RAW_CHUNKS_BY_NAME.values())):
        names = sorted(
            name for name, source in EXAMPLE_RAW_CHUNKS_BY_NAME.items() if source == raw_chunks
        )
        frames = extract_raw_world_frames(raw_chunks, [frame_indices[name] for name in names])
        frames_by_name.update(dict(zip(names, frames)))

    # Replace only TIFF outputs, preserving README.md and find_DGain.ipynb.
    output_dir = assert_safe_data_target(EXAMPLE_IMAGE_OUTPUT)
    output_dir.mkdir(parents=True, exist_ok=True)
    existing_tiffs = list(output_dir.glob("*.tiff"))
    if existing_tiffs and not overwrite:
        raise FileExistsError(
            f"{output_dir} already contains TIFFs. Set OVERWRITE_EXAMPLE_IMAGES to True to replace them."
        )
    if overwrite:
        # Explicit overwrite replaces curated images but never the support files.
        for existing_tiff in existing_tiffs:
            existing_tiff.unlink()

    # Write the raw arrays under their human-readable curated filenames.
    for filename, frame in sorted(frames_by_name.items()):
        output_path = output_dir / filename
        if not cv2.imwrite(str(output_path), frame, [cv2.IMWRITE_TIFF_COMPRESSION, 1]):
            raise IOError(f"OpenCV could not write {output_path}")
    print(f"Recreated {len(frames_by_name)} raw example images in {output_dir}")


if RUN_EXAMPLE_INDEX_DISCOVERY:
    discovered_example_indices = discover_reference_frame_indices(
        EXAMPLE_REFERENCE_PATHS, EXAMPLE_RAW_CHUNKS_BY_NAME
    )
    EXAMPLE_FRAME_INDICES.update(discovered_example_indices)
    update_example_frame_index_readme(discovered_example_indices)
    print("Discovered example indices:", discovered_example_indices)

if RUN_EXAMPLE_FRAME_EXTRACTION:
    unresolved = [name for name, index in EXAMPLE_FRAME_INDICES.items() if index is None]
    if unresolved:
        raise ValueError(f"Discover or enter the global raw indices for: {unresolved}")
    populate_example_world_camera_images(
        {name: int(index) for name, index in EXAMPLE_FRAME_INDICES.items()},
        overwrite=OVERWRITE_EXAMPLE_IMAGES,
    )

# `data/fisheyeLensCalibration/`

**What this is:** The checkerboard images and MATLAB Single Camera Calibrator session used to estimate focal length, principal point, and fisheye distortion for the ArduCam B0392 IMX219 world camera.

**Where it comes from:** A manually collected checkerboard-calibration raw recording followed by selection and fitting in MATLAB's Camera Calibrator app. The accepted local TIFFs are used once to discover their global raw-frame indices; thereafter the 47-image input set is recreated directly from the raw chunks.

**What it does:** Discovers and records the 47 selected raw indices, extracts those frames without a video conversion, and optionally opens the regenerated image directory in MATLAB Camera Calibrator. Image acceptance, model fitting, and saving `intrinsics_calibration_session.mat` remain interactive because those app decisions are part of the calibration measurement.

**Output:** `data/fisheyeLensCalibration/intrinsics_calibration_session.mat` and `data/fisheyeLensCalibration/intrinsics_calibration_images/`. After population, open the session with `cameraCalibrator('intrinsics_calibration_session.mat')`; the exported intrinsics belong in `derived/arducamB0392cameraInstrinsics.mat`.

In [ ]:
RUN_FISHEYE_INDEX_DISCOVERY = False
RUN_FISHEYE_FRAME_EXTRACTION = False
RUN_FISHEYE_CALIBRATOR = False
OVERWRITE_FISHEYE_IMAGES = False
FISHEYE_RAW_CHUNKS: Path | None = None  # Set to the exact checkerboard recording chunk directory.
FISHEYE_OUTPUT = DATA_ROOT / "fisheyeLensCalibration"
FISHEYE_IMAGE_OUTPUT = FISHEYE_OUTPUT / "intrinsics_calibration_images"
FISHEYE_IMAGE_NAMES = [f"{index}.tiff" for index in range(47)]
FISHEYE_REFERENCE_PATHS = {name: FISHEYE_IMAGE_OUTPUT / name for name in FISHEYE_IMAGE_NAMES}
# Populate this mapping with the discovery result to make later runs independent
# of the already-extracted reference TIFFs.
FISHEYE_FRAME_INDICES: dict[str, int | None] = {name: None for name in FISHEYE_IMAGE_NAMES}


def update_fisheye_frame_index_readme(frame_indices: dict[str, int]) -> None:
    """Record the accepted checkerboard frames' global raw indices."""
    rows = [
        "| Calibration image | Zero-based global raw-frame index |",
        "| --- | ---: |",
    ]
    for name in FISHEYE_IMAGE_NAMES:
        rows.append(f"| `{name}` | {frame_indices[name]} |")
    update_generated_readme_section(
        FISHEYE_OUTPUT / "README.md", "fisheye-frame-indices", "\n".join(rows)
    )


def populate_fisheye_calibration_images(
    raw_chunks: Path, frame_indices: dict[str, int], *, overwrite: bool = False
) -> None:
    """Recreate the accepted checkerboard images directly from raw chunks."""
    if list(frame_indices) != FISHEYE_IMAGE_NAMES:
        raise ValueError("Fisheye indices must be ordered from 0.tiff through 46.tiff.")
    # Preserve selection order so write_tiff_stack recreates the expected filenames.
    frames = extract_raw_world_frames(raw_chunks, list(frame_indices.values()))
    write_tiff_stack(frames, FISHEYE_IMAGE_OUTPUT, overwrite=overwrite)


if RUN_FISHEYE_INDEX_DISCOVERY:
    if FISHEYE_RAW_CHUNKS is None:
        raise ValueError("Set FISHEYE_RAW_CHUNKS before discovering frame indices.")
    discovered_fisheye_indices = discover_reference_frame_indices(
        FISHEYE_REFERENCE_PATHS,
        {name: FISHEYE_RAW_CHUNKS for name in FISHEYE_IMAGE_NAMES},
    )
    FISHEYE_FRAME_INDICES.update(discovered_fisheye_indices)
    update_fisheye_frame_index_readme(discovered_fisheye_indices)
    print("Discovered fisheye indices:", discovered_fisheye_indices)

if RUN_FISHEYE_FRAME_EXTRACTION:
    if FISHEYE_RAW_CHUNKS is None:
        raise ValueError("Set FISHEYE_RAW_CHUNKS before extracting calibration images.")
    unresolved = [name for name, index in FISHEYE_FRAME_INDICES.items() if index is None]
    if unresolved:
        raise ValueError(f"Discover or enter the fisheye frame indices for: {unresolved}")
    populate_fisheye_calibration_images(
        FISHEYE_RAW_CHUNKS,
        {name: int(index) for name, index in FISHEYE_FRAME_INDICES.items()},
        overwrite=OVERWRITE_FISHEYE_IMAGES,
    )

if RUN_FISHEYE_CALIBRATOR:
    # This intentionally opens MATLAB's interactive app; save the resulting session
    # as data/fisheyeLensCalibration/intrinsics_calibration_session.mat.
    import matlab.engine
    matlab_engine = matlab.engine.start_matlab()
    matlab_engine.cameraCalibrator(str(FISHEYE_IMAGE_OUTPUT), nargout=0)

# `data/flatFieldingFunction/`

**What this is:** Raw Bayer frames used to estimate the spatial sensitivity imposed by the fisheye lens. The camera pointed at the nominally uniform Fels Planetarium dome and was rotated about its optical axis. Averaging frames across orientations reduces dome-specific spatial structure while preserving camera/lens structure.

**Where it comes from:** World-camera chunks stored in the lab Dropbox under `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/fielding_function/planetarium_fielding_function_raw`. These are the paths and selections recovered from `code/library/matlabIO/python_libraries/scratch2.ipynb`.

**What it does:** Converts the chunks to a temporary grayscale video with digital gain applied, verifies the documented 180 fps frame rate, and extracts the 36 time samples currently consumed by `defineFlatFieldingFunction.m`. Files are named sequentially in selection order, not by original video-frame number.

**Output:** `data/flatFieldingFunction/rawFrames/0.tiff` through `35.tiff`. `defineFlatFieldingFunction.m` linearizes and averages these frames, fits the flattened Gaussian, and writes `derived/flatFieldingFunction.mat`.

In [ ]:
RUN_FLAT_FIELD_EXTRACTION = False
OVERWRITE_FLAT_FIELD_FRAMES = False
FLAT_FIELD_RAW_CHUNKS = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "fielding_function/planetarium_fielding_function_raw"
)
FLAT_FIELD_OUTPUT = DATA_ROOT / "flatFieldingFunction" / "rawFrames"

FLAT_FIELD_EXPECTED_FPS = 180.0
FLAT_FIELD_TIMES_SECONDS = [
    366, 372, 393, 401, 427, 435, 456, 466, 488, 498, 518, 528, 550, 560,
    582, 592, 614, 624, 646, 656, 678, 686, 708, 718, 742, 752, 774, 784,
    806, 814, 836, 846, 868, 878, 912, 920,
]
FLAT_FIELD_FRAME_INDICES = [
    65880, 66960, 70740, 72180, 76860, 78300, 82080, 83880, 87840,
    89640, 93240, 95040, 99000, 100800, 104760, 106560, 110520, 112320,
    116280, 118080, 122040, 123480, 127440, 129240, 133560, 135360,
    139320, 141120, 145080, 146520, 150480, 152280, 156240, 158040,
    164160, 165600,
]
assert [round(time * FLAT_FIELD_EXPECTED_FPS) for time in FLAT_FIELD_TIMES_SECONDS] == FLAT_FIELD_FRAME_INDICES

In [ ]:
def populate_flat_field_frames(raw_chunks: Path, *, overwrite: bool = False) -> None:
    """Extract the documented 36 planetarium orientations from raw chunks."""
    # Load video_io only for an enabled extraction and reject a stale source path early.
    video_io = load_video_io()
    raw_chunks = raw_chunks.resolve()
    if not raw_chunks.is_dir():
        raise FileNotFoundError(raw_chunks)

    # The AVI is an intermediate only. Keeping it in a temporary directory avoids
    # creating a second large calibration artifact in either Dropbox or data/.
    with tempfile.TemporaryDirectory(prefix="populate_flat_field_") as temporary_dir:
        temporary_video = Path(temporary_dir) / "planetarium_fielding_function.avi"
        # Reconstruct the contiguous recording and apply its per-frame digital gain,
        # matching the preprocessing used for the checked-in flat-field TIFFs.
        video_io.world_chunks_to_video(
            str(raw_chunks),
            output_path=str(temporary_video),
            verbose=True,
            convert_to_seconds=True,
            fill_missing_frames=True,
            apply_digital_gain=True,
        )
        # The documented times map to fixed indices only at the original 180 fps.
        frames_per_second = float(video_io.inspect_video_FPS(str(temporary_video)))
        if not np.isclose(frames_per_second, FLAT_FIELD_EXPECTED_FPS):
            raise ValueError(
                f"Expected a {FLAT_FIELD_EXPECTED_FPS:g} fps flat-field video; "
                f"found {frames_per_second:g} fps. Review the documented frame selection."
            )
        # Convert the human-readable time selection to video indices and cross-check
        # it against the authoritative list in defineFlatFieldingFunction.m.
        frame_indices = [round(time * frames_per_second) for time in FLAT_FIELD_TIMES_SECONDS]
        if frame_indices != FLAT_FIELD_FRAME_INDICES:
            raise ValueError("The derived flat-field indices no longer match defineFlatFieldingFunction.m.")

        # Extract in selection order and store the results directly as 0.tiff-35.tiff.
        frames = video_io.extract_frames_from_video(
            str(temporary_video), frame_indices, verbose=True, is_grayscale=True
        )
        write_tiff_stack(frames, FLAT_FIELD_OUTPUT, overwrite=overwrite)


if RUN_FLAT_FIELD_EXTRACTION:
    populate_flat_field_frames(
        FLAT_FIELD_RAW_CHUNKS, overwrite=OVERWRITE_FLAT_FIELD_FRAMES
    )

# `data/radiometricCorrectionRGB/`

**What this is:** A paired calibration between the spectral radiance of a cloudy sky measured with a PR670 and raw IMX219 images of that same sky. It is used to derive multiplicative RGB radiometric weights.

**Where it comes from:** The world-camera chunks are in Dropbox under `FLIC_data/LightLoggerRadCal/W1P1M1/radiometricCorrectionRGB/cloudyDayRecording`. The notebook logic comes from `code/preprocessRecordingData/another_scratch.ipynb`. The PR670 file `CloudySkySPD_37degSolarElevation.mat` and the illustrative `cropExample.tiff` are separately archived measurement inputs; this notebook does not recreate them.

**What it does:** Builds a temporary video without digital-gain, response-linearization, or color-weight corrections, then extracts grayscale frames 8000 through 8009. Preserving the uncorrected sensor values is essential because the downstream calibration is estimating those corrections.

**Output:** `data/radiometricCorrectionRGB/rawFrames/0.tiff` through `9.tiff`. `defineRadiometricWeights.m` combines these frames with the PR670 SPD and writes `derived/radiometricCorrectionRGB.mat`.

In [ ]:
RUN_RADIOMETRIC_FRAME_EXTRACTION = False
OVERWRITE_RADIOMETRIC_FRAMES = False
RADIOMETRIC_RAW_CHUNKS = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_data/LightLoggerRadCal/W1P1M1/"
    "radiometricCorrectionRGB/cloudyDayRecording"
)
RADIOMETRIC_OUTPUT = DATA_ROOT / "radiometricCorrectionRGB" / "rawFrames"
RADIOMETRIC_FRAME_INDICES = list(range(8000, 8010))


def populate_radiometric_frames(raw_chunks: Path, *, overwrite: bool = False) -> None:
    """Extract ten uncorrected cloudy-sky frames for RGB radiometric fitting."""
    # Defer video dependencies until use and verify the configured chunk archive exists.
    video_io = load_video_io()
    raw_chunks = raw_chunks.resolve()
    if not raw_chunks.is_dir():
        raise FileNotFoundError(raw_chunks)

    # Use a disposable AVI so the reconstructed recording never becomes a data asset.
    with tempfile.TemporaryDirectory(prefix="populate_radiometric_") as temporary_dir:
        temporary_video = Path(temporary_dir) / "cloudy_day_recording.avi"
        # Disable every camera correction because these frames are inputs used to
        # estimate those corrections downstream.
        video_io.world_chunks_to_video(
            str(raw_chunks),
            str(temporary_video),
            apply_digital_gain=False,
            convert_to_seconds=True,
            fill_missing_frames=True,
            verbose=True,
            linearize_camera_responsivity=False,
            apply_color_weights=False,
        )
        # Preserve the Bayer mosaic as grayscale and rename frames by selection order.
        frames = video_io.extract_frames_from_video(
            str(temporary_video), RADIOMETRIC_FRAME_INDICES, is_grayscale=True
        )
        write_tiff_stack(frames, RADIOMETRIC_OUTPUT, overwrite=overwrite)


if RUN_RADIOMETRIC_FRAME_EXTRACTION:
    populate_radiometric_frames(
        RADIOMETRIC_RAW_CHUNKS, overwrite=OVERWRITE_RADIOMETRIC_FRAMES
    )

# `data/darkSignal/`

**What this is:** Raw Bayer dark frames acquired with the lens cap installed, the camera wrapped in a black shroud, and the room dark. Separate recordings cover five fixed AGC states because exposure and analog gain can change the camera's dark behavior.

**Where it comes from:** The canonical recordings are stored in Dropbox at `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/darkNoiseCalibrations`, which is the configured source below. Each child directory must be one raw world-camera chunk recording and should retain its metadata-rich `AGCstate*_AGain-*_DGain-*_E-*` name.

**What it does:** Counts the raw frames physically present in each state recording, chooses ten global raw indices evenly across that count, and extracts those Bayer arrays directly from their chunk files. The selected indices are written into this measurement's README. No video conversion, gain, response linearization, or color weighting is applied.

**Output:** One ten-frame directory per state beneath `data/darkSignal/`. The README is preserved. `defineDarkSignal.m` subsequently computes the median dark signal and writes `derived/darkSignal.mat`.

In [ ]:
RUN_DARK_SIGNAL_EXTRACTION = False
OVERWRITE_DARK_SIGNAL_FRAMES = False
DARK_SIGNAL_RAW_ROOT = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "darkNoiseCalibrations"
)
DARK_SIGNAL_OUTPUT = DATA_ROOT / "darkSignal"
DARK_SIGNAL_FRAME_COUNT = 10
DARK_SIGNAL_STATE_PATTERN = re.compile(r"^AGCstate[1-5](?:_|$)")


def populate_dark_signal_frames(raw_root: Path, *, overwrite: bool = False) -> None:
    """Extract ten uncorrected dark frames from each of five fixed AGC states."""
    # Treat raw_root as the parent of the five state recordings and fail early if absent.
    raw_root = raw_root.resolve()
    if not raw_root.is_dir():
        raise FileNotFoundError(raw_root)

    # Select only directories whose names encode one of the expected AGC states.
    state_recordings = sorted(
        path
        for path in raw_root.iterdir()
        if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)
    )
    if len(state_recordings) != 5:
        raise ValueError(
            f"Expected five AGC-state recording directories under {raw_root}; found {len(state_recordings)}."
        )

    selected_indices_by_state: dict[str, list[int]] = {}
    # Process each state independently and read only the ten selected raw arrays.
    for recording_path in state_recordings:
        # Spanning 0 through count-1 samples the full acquisition uniformly.
        recording_frame_count = raw_world_frame_count(recording_path)
        frame_indices = np.linspace(
            0, recording_frame_count - 1, DARK_SIGNAL_FRAME_COUNT, dtype=np.int64
        ).tolist()
        frames = extract_raw_world_frames(recording_path, frame_indices)
        selected_indices_by_state[recording_path.name] = frame_indices
        # Keep each AGC state in its own metadata-rich output directory.
        write_tiff_stack(
            frames, DARK_SIGNAL_OUTPUT / recording_path.name, overwrite=overwrite
        )

    # Persist the exact selection produced from each recording's raw frame count.
    rows = [
        "| AGC-state folder | Zero-based global raw-frame indices |",
        "| --- | --- |",
    ]
    for state_name, frame_indices in selected_indices_by_state.items():
        rows.append(f"| `{state_name}` | {', '.join(map(str, frame_indices))} |")
    update_generated_readme_section(
        DARK_SIGNAL_OUTPUT / "README.md", "dark-signal-frame-indices", "\n".join(rows)
    )


if RUN_DARK_SIGNAL_EXTRACTION:
    populate_dark_signal_frames(
        DARK_SIGNAL_RAW_ROOT, overwrite=OVERWRITE_DARK_SIGNAL_FRAMES
    )

# Root-level files in `data/`

The top level of `data/` contains several MAT files rather than another directory. Only the AGC-to-illuminance file came from one of the consolidated Python notebooks.

## `empircalAGCAndIlluminance.mat`

**Source:** Raw `GKA` recordings from the 2026 scripted indoor/outdoor dataset mounted at `/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026`.

**Operation:** Run `deriveAGCLag.py` to write the shared lag under `derived/`, then process the raw recordings in memory with that fixed lag, discard each recording's initial transient samples, retain finite positive samples below the configured saturation limit, and write the matched linear-scale camera-score/illuminance point cloud.

**Output:** `deriveEmpircalAGCAndIlluminance.py` reads `derived/cameraAGCLag.mat` and writes `data/empircalAGCAndIlluminance.mat` directly. The file contains the MATLAB struct `empiralAGC` with the fields `cameraScoreLinear`, `msIlluminance`, and `sharedLagSeconds`. The later MATLAB piecewise log-log fit is model fitting, not data population, so it is not run here.

In [ ]:
RUN_AGC_TO_ILLUMINANCE = False
OVERWRITE_AGC_TO_ILLUMINANCE = False
AGC_RAW_ROOT = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")
AGC_MAXIMUM_SUBJECTS = 4
AGC_SUBJECTS_TO_SKIP = {"FLIC_18"}
AGC_MAXIMUM_SATURATION_PERCENT = 40.0
AGC_INITIAL_SAMPLES_TO_EXCLUDE = 100
AGC_DATA_OUTPUT = DATA_ROOT / "empircalAGCAndIlluminance.mat"
AGC_LAG_OUTPUT = PROJECT_ROOT / "derived" / "cameraAGCLag.mat"


def natural_sort_key(path_or_name: Path | str) -> list[int | str]:
    """Split names into text and integer pieces so FLIC_2 sorts before FLIC_10."""
    # Numeric tokens become integers while text tokens compare case-insensitively.
    return [int(piece) if piece.isdigit() else piece.lower() for piece in re.split(r"(\d+)", str(path_or_name))]


def populate_agc_to_illuminance(raw_root: Path, *, overwrite: bool = False) -> None:
    """Rebuild the empirical AGC/illuminance MAT file from raw subject recordings."""
    # Validate the mounted dataset before importing or running the heavier analysis.
    raw_root = raw_root.resolve()
    if not raw_root.is_dir():
        raise FileNotFoundError(raw_root)

    # Import the canonical derivation module from beside this notebook so there is
    # only one implementation of the point selection and MAT export logic.
    derive_module_dir = PROJECT_ROOT / "code" / "defineWorldCameraCalibration"
    data_prep_module_dir = derive_module_dir / "dataPrep"
    for import_path in (derive_module_dir, data_prep_module_dir):
        if str(import_path) not in sys.path:
            sys.path.insert(0, str(import_path))
    import deriveAGCLag
    import deriveEmpircalAGCAndIlluminance
    importlib.reload(deriveAGCLag)
    importlib.reload(deriveEmpircalAGCAndIlluminance)

    # Collect every activity/GKA recording for the first configured valid subjects.
    recording_paths: list[str] = []
    valid_subject_count = 0
    for subject_dir in sorted(raw_root.iterdir(), key=natural_sort_key):
        if valid_subject_count >= AGC_MAXIMUM_SUBJECTS:
            break
        # Ignore hidden/non-subject entries and any explicitly excluded subjects.
        if (
            not subject_dir.is_dir()
            or subject_dir.name.startswith(".")
            or subject_dir.name in AGC_SUBJECTS_TO_SKIP
        ):
            continue
        # Each activity contributes its GKA directory as one analysis recording.
        for activity_dir in sorted(subject_dir.iterdir(), key=natural_sort_key):
            if not activity_dir.is_dir() or activity_dir.name.startswith("."):
                continue
            recording_path = activity_dir / "GKA"
            if not recording_path.is_dir():
                raise FileNotFoundError(recording_path)
            recording_paths.append(str(recording_path))
        valid_subject_count += 1

    # Do not replace the curated MAT output without an explicit overwrite request.
    if AGC_DATA_OUTPUT.exists() and not overwrite:
        raise FileExistsError(AGC_DATA_OUTPUT)

    # Derive the shared lag first, then process the recordings with that lag
    # and export the selected camera-score/illuminance point cloud.
    lag_result = deriveAGCLag.derive_agc_lag(
        recording_paths, output_path=AGC_LAG_OUTPUT
    )
    deriveEmpircalAGCAndIlluminance.derive_empircal_agc_and_illuminance(
        recording_paths,
        lag_path=lag_result.output_path,
        output_path=AGC_DATA_OUTPUT,
        maximum_saturation_percent=AGC_MAXIMUM_SATURATION_PERCENT,
        initial_samples_to_exclude=AGC_INITIAL_SAMPLES_TO_EXCLUDE,
    )
    print(f"Generated AGC-to-illuminance data at {AGC_DATA_OUTPUT}")


if RUN_AGC_TO_ILLUMINANCE:
    populate_agc_to_illuminance(
        AGC_RAW_ROOT, overwrite=OVERWRITE_AGC_TO_ILLUMINANCE
    )

## Other root-level reference and calibration MAT files

- `ASM7341_spectralSensitivity.mat` is a reference table transcribed from the manufacturer-supplied `AS7341_Filter_Templates.xlsx` spreadsheet.
- `IMX219_spectralSensitivity.mat` is a reference table corresponding to Figure 18 of Pagnutti et al. (2017), supplied by the paper's first author.
- `camera_linearity_ND0_ND0p4_rgb_means.mat` contains the ND 0 and ND 0.4 RGB means used to fit the full-well-capacity effect. The recordings are collected with `collect_light_logger_calibration_data.m`, parsed by `convert_light_logger_calibration_data.m` with both `use_mean_frame` and `differentiate_color` enabled so the Bayer channels remain separate, and then saved by `analyze_camera_linearity_data.m`. The analysis writes into MATLAB's current directory, so the result should be reviewed before being placed in `data/`.

These files should remain curated calibration inputs rather than being silently overwritten by this notebook.

# Validate the populated data tree

This read-only audit checks the expected counts and key files after any population sections have run. It deliberately ignores deprecated artifacts, hidden Finder metadata, and Python cache files.

In [ ]:
def numbered_tiff_count(directory: Path) -> int:
    """Count sequential data TIFFs while ignoring README and hidden support files."""
    # Only numeric stems belong to the zero-based frame stacks generated above.
    return len([path for path in directory.glob("*.tiff") if path.stem.isdigit()])


checks = {
    "five curated example images": {
        path.name for path in (DATA_ROOT / "exampleWorldCameraImages").glob("*.tiff")
    } == EXPECTED_EXAMPLE_NAMES,
    "example-image README": (DATA_ROOT / "exampleWorldCameraImages" / "README.md").is_file(),
    "example-image DGain notebook": (DATA_ROOT / "exampleWorldCameraImages" / "find_DGain.ipynb").is_file(),
    "fisheye session": (DATA_ROOT / "fisheyeLensCalibration" / "intrinsics_calibration_session.mat").is_file(),
    "fisheye images": numbered_tiff_count(DATA_ROOT / "fisheyeLensCalibration" / "intrinsics_calibration_images") == 47,
    "36 flat-field frames": numbered_tiff_count(FLAT_FIELD_OUTPUT) == 36,
    "radiometric frames": numbered_tiff_count(RADIOMETRIC_OUTPUT) == 10,
    "radiometric README": (DATA_ROOT / "radiometricCorrectionRGB" / "README.md").is_file(),
    "five dark-signal states": len([path for path in DARK_SIGNAL_OUTPUT.iterdir() if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)]) == 5,
    "ten frames in every dark-signal state": all(
        numbered_tiff_count(path) == 10
        for path in DARK_SIGNAL_OUTPUT.iterdir()
        if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)
    ),
    "AGC-to-illuminance MAT": AGC_DATA_OUTPUT.is_file(),
    "AS7341 sensitivity MAT": (DATA_ROOT / "ASM7341_spectralSensitivity.mat").is_file(),
    "IMX219 sensitivity MAT": (DATA_ROOT / "IMX219_spectralSensitivity.mat").is_file(),
    "camera-linearity MAT": (DATA_ROOT / "camera_linearity_ND0_ND0p4_rgb_means.mat").is_file(),
}

for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {description}")

if not all(checks.values()):
    raise AssertionError("One or more expected calibration-data checks failed.")